In [1]:
import folium
from folium.plugins import Fullscreen
import geopandas as gpd

In [2]:
import black
import jupyter_black

jupyter_black.load(
    line_length=78,
    target_version=black.TargetVersion.PY310,
)

In [3]:
gradient_green = ["#006d2c", "#31a354", "#a1d99b"]  # green
gradient_teal = ["#006d77", "#83c5be", "#edf6f9"]  # teal-gold
gradient_orange = ["#a63603", "#e6550d", "#fdae6b"]  # orange-red
gradient_blue = ["#08519c", "#3182bd", "#bdd7e7"]  # blue

display_color = gradient_green
comfort_levels = ["high", "medium", "low"]

display_color_map = dict(zip(comfort_levels, display_color))

comfort_map = {
    "Neighborhood Greenway": "high",
    "Shared Use Path": "high",
    "Protected Bikeway": "high",
    "Accessway": "high",
    "Route": "medium",
    "Sidewalk Path": "high",
    "Buffered Bike Lane": "medium",
    "Shared Lane": "medium",
    "Paved Shoulder": "low",
    "Bike Lane": "low",
    "Routing Connection": "low",
}

In [4]:
gdf = gpd.read_file("data/city-of-eugene/Eugene_Bikeways_-_HUB.geojson")

In [5]:
gdf["comfort_level"] = gdf["ftypedes"].map(comfort_map)
high_only = gdf[gdf["comfort_level"] == "high"]
high_medium = gdf[gdf["comfort_level"].isin(["high", "medium"])]
all_routes = gdf.copy()

center = [
    gdf.geometry.union_all().centroid.y,
    gdf.geometry.union_all().centroid.x,
]

In [6]:
help_html = """
<div style="
    position: fixed; 
    bottom: 40px;
    left: 40px;
    width: 260px;
    z-index:9999;
    background-color: white;
    border-radius: 8px;
    padding: 10px;
    box-shadow: 0 0 8px rgba(0,0,0,0.2);
    font-size:14px;
">
<p>
<a href="https://github.com/nbirnel/atlas-of-eugene">Source</a>
</p><br>
Hover over routes for details.<br>This map is scrollable and zoomable.
</div>
"""

In [7]:
legend_html = f"""
<div style="
     position: fixed;
     bottom: 40px;
     right: 40px;
     width: 180px;
     z-index:9999;
     background-color: white;
     border-radius: 8px;
     padding: 10px;
     box-shadow: 0 0 6px rgba(0,0,0,0.3);
     font-size: 14px;">
<b>Comfort Level</b><br>
<i style="background:{display_color[0]};width:18px;height:3px;display:inline-block;margin-right:6px;"></i>High<br>
<i style="background:{display_color[1]};width:18px;height:3px;display:inline-block;margin-right:6px;"></i>Medium<br>
<i style="background:{display_color[2]};width:18px;height:3px;display:inline-block;margin-right:6px;"></i>Low
</div>
"""

In [8]:
m = folium.Map(location=center, zoom_start=12, tiles=None)
folium.TileLayer("CartoDB positron", control=False).add_to(m)


# Helper to create styled GeoJSON layers
def make_layer(subset, name):
    return folium.GeoJson(
        subset,
        name=name,
        style_function=lambda feature: style_map.get(
            "color": display_color_map[feature["properties"]["comfort_level"]]
            "weight": 4,
            "opacity": 0.8,
        ),
        tooltip=folium.GeoJsonTooltip(
            fields=["name", "ftypedes", "comfort_level"],
            aliases=["Route:", "Description:", "Comfort:"],
            localize=True,
        ),
        overlay=True,
        control=True,
    )


# Layers
layer_high = make_layer(high_only, "High Comfort")
layer_high_medium = make_layer(high_medium, "High + Medium Comfort")
layer_all = make_layer(all_routes, "All Routes")

# Order of addition controls what order is displayed in the Layout Control
layer_high.add_to(m)
layer_high_medium.add_to(m)
layer_all.add_to(m)


folium.LayerControl(collapsed=False, position="topright").add_to(m)
Fullscreen(position="topright").add_to(m)

# --- JS patch: make overlays radio-style ---
radio_js = """
<script>
document.addEventListener("DOMContentLoaded", function() {
  const overlayInputs = document.querySelectorAll('.leaflet-control-layers-overlays input[type="checkbox"]');
  overlayInputs.forEach(input => { input.type = 'radio'; input.name = 'comfort_layers'; });

  // Default: only keep "High + Medium Comfort" checked
  overlayInputs.forEach(input => {
    if (input.nextSibling.textContent.includes("High + Medium")) {
      if (!input.checked) input.click();
    } else if (input.checked) {
      input.click();
    }
  });
});
</script>
"""
m.get_root().html.add_child(folium.Element(radio_js))

title = "Eugene Bike Routes by Comfort Level"
title_html = (
    f'<h1 style="position:absolute;z-index:100000;left:20vw" >{title}</h1>'
)
m.get_root().html.add_child(folium.Element(title_html))
m.get_root().html.add_child(folium.Element(legend_html))
m.get_root().html.add_child(folium.Element(help_html))
m.save("bike-routes-by-comfort-level.html")
m

SyntaxError: invalid syntax (1563744410.py, line 11)

In [ ]:
m = folium.Map(location=center, zoom_start=12, tiles=None)
folium.TileLayer("CartoDB positron", control=False).add_to(m)


layers = []
for comfort in ["high", "medium", "low"]:
    subset_level = gdf[gdf["comfort_level"] == comfort]
    ftypedes_values = subset_level["ftypedes"].unique()

    for ftype in ftypedes_values:
        subset = subset_level[subset_level["ftypedes"] == ftype]
        layer_name = f"{comfort.capitalize()}: {ftype}"

        show_layer = comfort in ["high", "medium"]

        layer = folium.GeoJson(
            subset,
            name=layer_name,
            style_function=lambda feature, c=comfort: {
                "color": display_color_map[c],
                "weight": 4,
                "opacity": 0.8,
            },
            tooltip=folium.GeoJsonTooltip(
                fields=["name", "ftypedes"],
                aliases=["Route:", "Description:"],
                localize=True,
            ),
            overlay=True,
            control=True,
            show=show_layer,
        )

        # Add all layers to the map, but we will hide Low via JS
        layer.add_to(m)
        layers.append(layer)

# --- Layer control (checkboxes) ---
folium.LayerControl(collapsed=False, position="topright").add_to(m)
Fullscreen(position="topright").add_to(m)

collapsible_js = """
<script>
document.addEventListener("DOMContentLoaded", function() {
    const overlays = document.querySelector('.leaflet-control-layers-overlays');
    if (!overlays) return;

    const comfortLevels = ['High', 'Medium', 'Low'];
    const groupDivs = {};

    // Create a container for each comfort level
    comfortLevels.forEach(level => {
        const div = document.createElement('div');
        div.style.marginBottom = '4px';
        const header = document.createElement('b');
        header.textContent = level;
        header.style.cursor = 'pointer';
        header.style.display = 'block';
        header.style.margin = '4px 0';
        div.appendChild(header);

        const list = document.createElement('div');
        list.style.marginLeft = '10px';
        list.style.display = 'block';
        div.appendChild(list);

        overlays.appendChild(div);
        groupDivs[level] = list;

        // Toggle collapse on header click
        header.onclick = () => {
            list.style.display = list.style.display === 'none' ? 'block' : 'none';
        };
    });

    // Move each overlay into the right group
    const inputs = overlays.querySelectorAll('label');
    inputs.forEach(label => {
        const text = label.textContent.trim();
        comfortLevels.forEach(level => {
            if (text.startsWith(level + ":")) {
                groupDivs[level].appendChild(label);
            }
        });
    });
});
</script>
"""

m.get_root().html.add_child(folium.Element(collapsible_js))
title = "Eugene Bike Routes by Comfort and Type"
title_html = (
    f'<h1 style="position:absolute;z-index:100000;left:20vw" >{title}</h1>'
)
m.get_root().html.add_child(folium.Element(title_html))
m.get_root().html.add_child(folium.Element(help_html))
m.get_root().html.add_child(folium.Element(legend_html))
m.save("bike-routes-by-comfort-and-type.html")
m